# 💓 Notebook 1: Fixed-Interval Heartbeats

**The big question:** *"How does one machine know that another machine is still alive?"*

The basic idea is simple: every node sends a small "I'm alive!" message — a **heartbeat** — every few seconds. If a peer doesn't hear a heartbeat for some timeout window, it declares the sender dead.

But this is full of trade-offs:

- Short timeout → fast failure detection, **lots of false positives** (just a slow network looks like a dead node).
- Long timeout → reliable, **slow** to react to real failures.

In this notebook we simulate a heartbeat sender and a monitor, and see what happens with different timeouts. In notebook 2 we'll fix the trade-off with the **phi accrual** detector.

## Learning objectives
- Implement send/receive heartbeats with timestamps.
- Visualize the false-positive vs detection-time trade-off.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/heartbeat
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook).

In [ ]:
import random, time
random.seed(0)

# Simulate 60 seconds of heartbeats from a node that sends every 1.0s on average,
# with some jitter. The node really dies at t=30.
TICK = 1.0
JITTER = 0.4              # network jitter in seconds
DEAD_AT = 30.0
TOTAL = 60.0

heartbeats = []
t = 0.0
while t < TOTAL:
    t += TICK + random.uniform(-JITTER, JITTER)
    if t >= DEAD_AT:
        break
    heartbeats.append(t)

print(f"node sent {len(heartbeats)} heartbeats, last one at t={heartbeats[-1]:.2f}s, died at t={DEAD_AT}")

In [ ]:
def detect_dead(heartbeats, timeout, sample_every=0.1, total=TOTAL):
    '''Walk forward in time, declaring 'dead' when we haven't heard a beat for >timeout.'''
    last_seen = -float("inf")
    declared_dead_at = None
    false_positives = 0
    was_dead = False
    t = 0.0
    hb_iter = iter(heartbeats)
    next_hb = next(hb_iter, None)

    while t < total:
        while next_hb is not None and next_hb <= t:
            last_seen = next_hb
            if was_dead:
                false_positives += 1
                was_dead = False
            next_hb = next(hb_iter, None)
        if t - last_seen > timeout:
            if not was_dead:
                if declared_dead_at is None:
                    declared_dead_at = t
                was_dead = True
        t += sample_every
    return declared_dead_at, false_positives

for to in (0.5, 1.0, 1.5, 2.0, 3.0, 5.0):
    dead_at, fp = detect_dead(heartbeats, to)
    delay = (dead_at - DEAD_AT) if dead_at else None
    print(f"timeout={to}s -> declared dead at t={dead_at}, "
          f"detection delay={delay}, false positives during life={fp}")

## 🤔 What we just saw

- A short timeout (e.g. 0.5s) catches the failure quickly but flips between alive/dead during normal jitter — false positives that would page your on-call for nothing.
- A long timeout (5s) is stable but takes 5 seconds longer to notice a real failure.

There is no single "right" timeout. The next notebook fixes this with an **adaptive** detector that learns the network's normal jitter.